In [1]:
%pip install openpyxl duckdb

In [1]:
import pandas as pd
sheets = pd.read_excel("../data/online_retail_II.xlsx", sheet_name=None)
print({name: df.shape for name, df in sheets.items()})
df = pd.concat(sheets.values(), ignore_index=True)
print(df.shape)
print(df.columns.tolist())
df.head()

{'Year 2009-2010': (525461, 8), 'Year 2010-2011': (541910, 8)}
(1067371, 8)
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [2]:
from pathlib import Path

frames = []
for name, sheet in sheets.items():
    sheet = sheet.copy()
    sheet["source_sheet"] = name
    frames.append(sheet)

df = pd.concat(frames, ignore_index=True)

df["Invoice"] = df["Invoice"].astype(str)
df["StockCode"] = df["StockCode"].astype(str)

df.to_parquet(Path("../data") / "online_retail_II_raw.parquet", index=False)
print(df.shape)

ArrowTypeError: ("Expected bytes, got a 'int' object", 'Conversion failed for column Description with type object')

In [3]:
df["Description"].map(type).value_counts()

Description
<class 'str'>      1062985
<class 'float'>       4382
<class 'int'>            4
Name: count, dtype: int64

In [4]:
is_int = df["Description"].map(lambda x: isinstance(x, int))
df.loc[is_int, ["Invoice", "StockCode", "Description", "Quantity", "Price", "source_sheet"]].head(10)

,Invoice,StockCode,Description,Quantity,Price,source_sheet
6911,490007,84347,21494,-720,0.0,Year 2009-2010
274052,516016,22467,22719,2,0.0,Year 2009-2010
274053,516017,22719,22467,-2,0.0,Year 2009-2010
945851,572891,23343,20713,-400,0.0,Year 2010-2011


In [5]:
for col in df.select_dtypes(include="object").columns:
    print(col, df[col].map(type).value_counts().to_dict())

Invoice {<class 'str'>: 1067371}
StockCode {<class 'str'>: 1067371}
Description {<class 'str'>: 1062985, <class 'float'>: 4382, <class 'int'>: 4}
Country {<class 'str'>: 1067371}
source_sheet {<class 'str'>: 1067371}


In [6]:
df["Description"] = df["Description"].astype("string")
df.to_parquet(Path("../data") / "online_retail_II_raw.parquet", index=False)
print(df.shape)

(1067371, 9)
